# Amazon ML Challenge 2026: Stage 6 — Test Inference & Leaderboard Submission
### Official Deliverable Generator for `test_source1.tsv` (1,732,544 Entities)

**Purpose:** Generate the official, competition-valid submission files:
1. `matching_results.tsv` — The scored file uploaded to the competition portal.
2. `candidate_pairs.tsv`  — The blocking audit file required in the final archive.

**Inputs Attached in Kaggle:**
- `mlchallengedata` — Competition dataset containing `dataset/test/test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv`
- `ml1nbv2` — Normalized parquets (`test_s1_norm.parquet`, `test_s2_norm.parquet`, `test_s3_norm.parquet`)
- `ml4nbv1` — Pre-trained LightGBM model weights (`lgbm_model_fold0.txt`, `lgbm_model_fold1.txt`)
- `ml2nbv1` / `ml2nbv2` — Test dense embeddings (`test_s1_name_emb.npy`, etc.) if available

**Runtime & Architecture:**
- **Zero training / zero 50-minute grid searches:** Applies the winning thresholds ($t_{first}=0.600, t_{rest}=0.575$) discovered on the 84.7M training set.
- **5M Chunk Streaming:** Keeps host RAM usage flat at < 6 GB (no OOM).
- **GPU Accelerated:** Runs FAISS retrieval on T4 GPUs in ~5 minutes; entire notebook finishes in ~25 minutes.


In [ ]:
!pip install -q rapidfuzz lightgbm pyarrow faiss-gpu

import os, sys, gc, time, re, unicodedata, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import lightgbm as lgb
from rapidfuzz.distance import JaroWinkler

print("Libraries imported successfully.")
print(f"LightGBM version: {lgb.__version__}")


In [ ]:
# ============================================================
# ROBUST PATH DISCOVERY (Auto-detects Kaggle inputs)
# ============================================================

IS_KAGGLE = os.path.exists('/kaggle')

def find_file(pattern):
    search_root = Path('/kaggle/input') if IS_KAGGLE else Path('.')
    matches = list(search_root.rglob(pattern))
    if not matches:
        return None
    matches.sort(key=lambda p: (p.stat().st_size, p.stat().st_mtime), reverse=True)
    return matches[0]

def find_dir_by_file(pattern):
    f = find_file(pattern)
    return f.parent if f else None

WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('work')
OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Dataset TSVs
TEST_S1_TSV = find_file('test_source1.tsv')
TEST_S2_TSV = find_file('test_source2.tsv')
TEST_S3_TSV = find_file('test_source3.tsv')

# 2. Normalized files
TEST_S1_NORM = find_file('test_s1_norm.parquet')
TEST_S2_NORM = find_file('test_s2_norm.parquet')
TEST_S3_NORM = find_file('test_s3_norm.parquet')

# 3. Model weights
MODEL_FOLD0 = find_file('lgbm_model_fold0.txt')
MODEL_FOLD1 = find_file('lgbm_model_fold1.txt')

# 4. Dense embeddings (if present from Stage 1a)
TEST_S1_NAME_EMB = find_file('test_s1_name_emb.npy')
TEST_S2_NAME_EMB = find_file('test_s2_name_emb.npy')
TEST_S3_NAME_EMB = find_file('test_s3_name_emb.npy')

# 5. Validator
VALIDATOR_SCRIPT = find_file('validate_submission.py')

print("\n" + "=" * 60)
print(" DISCOVERED INPUT ARTIFACTS")
print("=" * 60)
print(f"  Test S1 TSV:      {TEST_S1_TSV}")
print(f"  Test S2 TSV:      {TEST_S2_TSV}")
print(f"  Test S3 TSV:      {TEST_S3_TSV}")
print(f"  Test S1 Norm:     {TEST_S1_NORM}")
print(f"  Model Fold 0:     {MODEL_FOLD0}")
print(f"  Model Fold 1:     {MODEL_FOLD1}")
print(f"  Test S1 Emb:      {TEST_S1_NAME_EMB}")
print(f"  Validator:        {VALIDATOR_SCRIPT}")
print(f"  Output Dir:       {OUTPUT_DIR}")
print("=" * 60)

assert TEST_S1_TSV is not None and TEST_S1_TSV.exists(), "ERROR: test_source1.tsv not found! Attach mlchallengedata."
assert MODEL_FOLD0 is not None and MODEL_FOLD0.exists(), "ERROR: lgbm_model_fold0.txt not found! Attach ml4nbv1."


In [ ]:
# ============================================================
# NORMALIZATION: LOAD PRE-PROCESSED OR RUN STREAMING FALLBACK
# ============================================================

US_STATE_CODES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA',
    'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT',
    'VA','WA','WV','WI','WY','DC'
}

def clean_text_fast(text):
    if not text or pd.isna(text):
        return ""
    text = unicodedata.normalize('NFKD', str(text))
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def extract_house_number(addr):
    if not addr or pd.isna(addr):
        return "", False
    s = str(addr)
    if '##' in s or '***' in s:
        return "", True
    m = re.search(r'\b(\d{1,6}[A-Za-z]?)\b', s)
    return m.group(1).lower() if m else "", False

def normalize_df_stream(df):
    clean_names = [clean_text_fast(x) for x in df['business_name'].values]
    raw_addrs = df['business_address'].fillna('').astype(str).values
    clean_addrs = [clean_text_fast(x) for x in raw_addrs]
    
    house_nums = []
    redacteds = []
    for a in raw_addrs:
        h, r = extract_house_number(a)
        house_nums.append(h)
        redacteds.append(r)
        
    state_codes = []
    for a in clean_addrs:
        sc = ""
        for token in a.split():
            if token.upper() in US_STATE_CODES:
                sc = token.upper()
                break
        state_codes.append(sc)
        
    countries = df['country'].fillna('UNKNOWN').astype(str).str.upper().values
    
    return pd.DataFrame({
        'entity_id': df['entity_id'].astype(str).values,
        'name_core': clean_names,
        'addr_core': clean_addrs,
        'house_number': house_nums,
        'is_house_redacted': redacteds,
        'state_code': state_codes,
        'country': countries,
    })

# Load or generate normalized parquets
NORM_WORK = WORK_DIR / 'normalized'
NORM_WORK.mkdir(parents=True, exist_ok=True)

test_norm_files = {
    's1': (TEST_S1_NORM, TEST_S1_TSV, NORM_WORK / 'test_s1_norm.parquet'),
    's2': (TEST_S2_NORM, TEST_S2_TSV, NORM_WORK / 'test_s2_norm.parquet'),
    's3': (TEST_S3_NORM, TEST_S3_TSV, NORM_WORK / 'test_s3_norm.parquet'),
}

final_norm_paths = {}

for key, (cached_path, raw_tsv, target_path) in test_norm_files.items():
    if cached_path and cached_path.exists():
        print(f"  [Cached] Using pre-computed {key.upper()}: {cached_path} ({cached_path.stat().st_size / 1e6:.1f} MB)")
        final_norm_paths[key] = cached_path
    elif target_path.exists():
        print(f"  [Found] Using local {key.upper()}: {target_path}")
        final_norm_paths[key] = target_path
    else:
        print(f"  [Normalizing] Streaming {raw_tsv.name} -> {target_path.name}...")
        t0 = time.time()
        chunks = []
        for c in pd.read_csv(raw_tsv, sep='\t', chunksize=250000):
            chunks.append(normalize_df_stream(c))
        df_full = pd.concat(chunks, ignore_index=True)
        df_full.to_parquet(target_path, index=False)
        print(f"    Wrote {len(df_full):,} records in {time.time()-t0:.1f}s")
        final_norm_paths[key] = target_path
        del chunks, df_full; gc.collect()

print("\nNormalization verification complete!")


In [ ]:
# ============================================================
# TEST BLOCKING: GENERATE CANDIDATE PAIRS FOR TEST_SOURCE1
# ============================================================

cand_parquet_path = WORK_DIR / 'candidates_test.parquet'
cand_tsv_path = OUTPUT_DIR / 'candidate_pairs.tsv'

# Check if candidates already exist
existing_cand_tsv = find_file('candidate_pairs.tsv')
if existing_cand_tsv and 'ml3' not in str(existing_cand_tsv) and existing_cand_tsv.stat().st_size > 100_000:
    print(f"Found existing test candidate pairs: {existing_cand_tsv}")
    shutil.copy2(existing_cand_tsv, cand_tsv_path)

# Verify if we need to run candidate retrieval
df_s1_test = pd.read_parquet(final_norm_paths['s1'], columns=['entity_id', 'name_core', 'country'])
s1_test_ids = df_s1_test['entity_id'].values
n_s1_test = len(s1_test_ids)
print(f"Total Test S1 Entities: {n_s1_test:,}")

# Check if FAISS dense embeddings exist for test
HAVE_FAISS_EMBS = (TEST_S1_NAME_EMB is not None and TEST_S1_NAME_EMB.exists() and
                   TEST_S2_NAME_EMB is not None and TEST_S3_NAME_EMB is not None)

if cand_parquet_path.exists():
    print(f"Using pre-generated candidates: {cand_parquet_path} ({cand_parquet_path.stat().st_size / 1e6:.1f} MB)")
elif HAVE_FAISS_EMBS:
    print("Found test dense embeddings! Running FAISS GPU Retrieval...")
    # FAISS GPU Retrieval Logic
    import faiss
    
    dim = 384
    top_k = 30
    nlist = 4096
    nprobe = 64
    
    # Load targets
    df_s2_ids = pd.read_parquet(final_norm_paths['s2'], columns=['entity_id'])
    df_s3_ids = pd.read_parquet(final_norm_paths['s3'], columns=['entity_id'])
    test_target_ids = np.concatenate([df_s2_ids['entity_id'].values, df_s3_ids['entity_id'].values])
    n_target_test = len(test_target_ids)
    del df_s2_ids, df_s3_ids; gc.collect()
    
    def run_faiss_channel(label, s2_path, s3_path, s1_path):
        print(f"  Building FAISS index for {label}...")
        arr_s2 = np.load(s2_path, mmap_mode='r')
        arr_s3 = np.load(s3_path, mmap_mode='r')
        n_s2 = len(arr_s2)
        n_s3 = len(arr_s3)
        
        quantizer = faiss.IndexFlatIP(dim)
        index_cpu = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        
        # Train on 200k samples
        train_idx = np.random.choice(n_s2, size=min(150000, n_s2), replace=False)
        index_cpu.train(np.asarray(arr_s2[train_idx], dtype=np.float32))
        
        # Transfer to GPU
        res = faiss.StandardGpuResources()
        res.setTempMemory(1024 * 1024 * 1024)
        index_gpu = faiss.index_cpu_to_gpu(res, 0, index_cpu)
        index_gpu.nprobe = nprobe
        
        # Add S2 and S3 in chunks
        for start in range(0, n_s2, 500000):
            end = min(start + 500000, n_s2)
            index_gpu.add(np.asarray(arr_s2[start:end], dtype=np.float32))
        for start in range(0, n_s3, 500000):
            end = min(start + 500000, n_s3)
            index_gpu.add(np.asarray(arr_s3[start:end], dtype=np.float32))
            
        print(f"    Index ready ({index_gpu.ntotal:,} vectors). Searching {n_s1_test:,} queries...")
        s1_embs = np.load(s1_path, mmap_mode='r')
        sims, indices = index_gpu.search(np.asarray(s1_embs, dtype=np.float32), top_k)
        
        del arr_s2, arr_s3, s1_embs, index_gpu, index_cpu, res; gc.collect()
        return indices, sims

    name_idx, name_sim = run_faiss_channel("Name", TEST_S2_NAME_EMB, TEST_S3_NAME_EMB, TEST_S1_NAME_EMB)
    
    # Write to Parquet in streaming batches
    schema = pa.schema([
        ('s1_id', pa.string()),
        ('candidate_id', pa.string()),
        ('sim_name', pa.float32()),
        ('sim_addr', pa.float32()),
        ('route_name', pa.int8()),
        ('route_addr', pa.int8()),
    ])
    
    writer = pq.ParquetWriter(cand_parquet_path, schema, compression='snappy')
    tsv_out = open(cand_tsv_path, 'w', encoding='utf-8')
    tsv_out.write("source1_entity_id\tcandidate_entity_ids\n")
    
    print("  Streaming FAISS candidates to disk...")
    batch_size = 50000
    for b_start in range(0, n_s1_test, batch_size):
        b_end = min(b_start + batch_size, n_s1_test)
        b_s1, b_cand, b_sn, b_sa, b_rn, b_ra = [], [], [], [], [], []
        
        for i in range(b_start, b_end):
            sid = s1_test_ids[i]
            row_cands = []
            for k in range(top_k):
                tid_idx = name_idx[i, k]
                if 0 <= tid_idx < n_target_test:
                    cid = test_target_ids[tid_idx]
                    row_cands.append(cid)
                    b_s1.append(sid)
                    b_cand.append(cid)
                    b_sn.append(float(name_sim[i, k]))
                    b_sa.append(0.0)
                    b_rn.append(1)
                    b_ra.append(0)
            tsv_out.write(f"{sid}\t{','.join(row_cands)}\n")
            
        tbl = pa.Table.from_arrays([
            pa.array(b_s1, type=pa.string()),
            pa.array(b_cand, type=pa.string()),
            pa.array(b_sn, type=pa.float32()),
            pa.array(b_sa, type=pa.float32()),
            pa.array(b_rn, type=pa.int8()),
            pa.array(b_ra, type=pa.int8()),
        ], schema=schema)
        writer.write_table(tbl)
        del tbl, b_s1, b_cand, b_sn, b_sa, b_rn, b_ra; gc.collect()
        
    writer.close()
    tsv_out.close()
    print("FAISS candidate generation complete!")
    del name_idx, name_sim; gc.collect()

else:
    print("Dense embeddings not found. Running high-throughput Multi-Key Lexical Blocking...")
    t0_lex = time.time()
    
    # Load target names and addresses
    df_s2 = pd.read_parquet(final_norm_paths['s2'], columns=['entity_id', 'name_core', 'addr_core', 'country'])
    df_s3 = pd.read_parquet(final_norm_paths['s3'], columns=['entity_id', 'name_core', 'addr_core', 'country'])
    df_tgt = pd.concat([df_s2, df_s3], ignore_index=True)
    del df_s2, df_s3; gc.collect()
    
    tgt_ids = df_tgt['entity_id'].values
    tgt_names = df_tgt['name_core'].values
    tgt_addrs = df_tgt['addr_core'].values
    n_tgt = len(tgt_ids)
    
    # Build 3 high-recall index tables (capped lists to keep memory flat under 2 GB)
    print(f"  Indexing {n_tgt:,} target records into multi-key index...")
    name_to_tgt_idx = {}
    tok2_to_tgt_idx = {}
    geo_to_tgt_idx = {}
    
    for idx in range(n_tgt):
        nm = tgt_names[idx]
        if nm and len(nm) >= 2:
            # 1. Exact clean name root
            lst = name_to_tgt_idx.get(nm)
            if lst is None:
                name_to_tgt_idx[nm] = [idx]
            elif len(lst) < 15:
                lst.append(idx)
                
            # 2. First 2 tokens prefix
            tokens = nm.split()
            if len(tokens) >= 2:
                tok2 = f"{tokens[0]} {tokens[1]}"
                lst2 = tok2_to_tgt_idx.get(tok2)
                if lst2 is None:
                    tok2_to_tgt_idx[tok2] = [idx]
                elif len(lst2) < 10:
                    lst2.append(idx)
                    
            # 3. Postal code / Zip anchor + first token
            addr = tgt_addrs[idx]
            if addr and tokens:
                # Find 5-digit US/FR or 6-digit IN postal code
                m_zip = re.search(r'\b(\d{5,6})\b', addr)
                if m_zip:
                    geo_key = f"{m_zip.group(1)}_{tokens[0]}"
                    lst_geo = geo_to_tgt_idx.get(geo_key)
                    if lst_geo is None:
                        geo_to_tgt_idx[geo_key] = [idx]
                    elif len(lst_geo) < 5:
                        lst_geo.append(idx)
                        
    print(f"  Multi-key index ready in {time.time()-t0_lex:.1f}s:")
    print(f"    - Exact Name Keys:     {len(name_to_tgt_idx):,}")
    print(f"    - First-2-Tokens Keys: {len(tok2_to_tgt_idx):,}")
    print(f"    - Geo Anchor Keys:     {len(geo_to_tgt_idx):,}")
    
    # Load S1 name and address
    df_s1_full = pd.read_parquet(final_norm_paths['s1'], columns=['entity_id', 'name_core', 'addr_core'])
    s1_names_all = df_s1_full['name_core'].values
    s1_addrs_all = df_s1_full['addr_core'].values
    del df_s1_full; gc.collect()
    
    # Streaming candidates to Parquet & TSV
    schema = pa.schema([
        ('s1_id', pa.string()),
        ('candidate_id', pa.string()),
        ('sim_name', pa.float32()),
        ('sim_addr', pa.float32()),
        ('route_name', pa.int8()),
        ('route_addr', pa.int8()),
    ])
    
    writer = pq.ParquetWriter(cand_parquet_path, schema, compression='snappy')
    tsv_out = open(cand_tsv_path, 'w', encoding='utf-8')
    tsv_out.write("source1_entity_id\tcandidate_entity_ids\n")
    
    batch_size = 100000
    total_pairs = 0
    
    for b_start in range(0, n_s1_test, batch_size):
        b_end = min(b_start + batch_size, n_s1_test)
        b_s1, b_cand, b_sn, b_sa, b_rn, b_ra = [], [], [], [], [], []
        
        for i in range(b_start, b_end):
            sid = s1_test_ids[i]
            s_name = s1_names_all[i]
            s_addr = s1_addrs_all[i]
            
            cand_indices = []
            
            # Query 1: Exact Name Root
            if s_name:
                cand_indices.extend(name_to_tgt_idx.get(s_name, []))
                
                # Query 2: First 2 tokens prefix
                s_toks = s_name.split()
                if len(s_toks) >= 2:
                    tok2 = f"{s_toks[0]} {s_toks[1]}"
                    cand_indices.extend(tok2_to_tgt_idx.get(tok2, []))
                    
                # Query 3: Postal anchor
                if s_addr and s_toks:
                    m_zip = re.search(r'\b(\d{5,6})\b', s_addr)
                    if m_zip:
                        geo_key = f"{m_zip.group(1)}_{s_toks[0]}"
                        cand_indices.extend(geo_to_tgt_idx.get(geo_key, []))
            
            # Deduplicate preserving order (capped at 25 candidates per entity)
            seen = set()
            unique_cands = []
            for j in cand_indices:
                if j not in seen:
                    seen.add(j)
                    unique_cands.append(j)
                    if len(unique_cands) >= 25:
                        break
                        
            c_ids = [tgt_ids[j] for j in unique_cands]
            tsv_out.write(f"{sid}\t{','.join(c_ids)}\n")
            
            for j in unique_cands:
                b_s1.append(sid)
                b_cand.append(tgt_ids[j])
                b_sn.append(1.0 if tgt_names[j] == s_name else 0.8)
                b_sa.append(0.5)
                b_rn.append(1)
                b_ra.append(0)
                
        tbl = pa.Table.from_arrays([
            pa.array(b_s1, type=pa.string()),
            pa.array(b_cand, type=pa.string()),
            pa.array(b_sn, type=pa.float32()),
            pa.array(b_sa, type=pa.float32()),
            pa.array(b_rn, type=pa.int8()),
            pa.array(b_ra, type=pa.int8()),
        ], schema=schema)
        writer.write_table(tbl)
        total_pairs += len(b_s1)
        del tbl, b_s1, b_cand, b_sn, b_sa, b_rn, b_ra; gc.collect()
        
    writer.close()
    tsv_out.close()
    print(f"Generated {total_pairs:,} high-recall candidate pairs in {cand_parquet_path}!")
    del name_to_tgt_idx, tok2_to_tgt_idx, geo_to_tgt_idx, df_tgt; gc.collect()
    del s1_names_all, s1_addrs_all; gc.collect()


In [ ]:
# ============================================================
# STREAMING PAIRWISE FEATURE ENGINEERING & LIGHTGBM SCORING
# ============================================================

print("\n" + "=" * 60)
print(" LIGHTGBM MODEL INFERENCE (STREAMED CHUNKS)")
print("=" * 60)

# 1. Load trained LightGBM models
models = []
for p in [MODEL_FOLD0, MODEL_FOLD1]:
    if p and p.exists():
        print(f"  Loading model checkpoint: {p}")
        models.append(lgb.Booster(model_file=str(p)))

assert len(models) > 0, "No trained LightGBM models found!"

# 2. Build flat parallel lookup arrays for test records (fast O(1) in C++)
print("  Building parallel record lookup arrays...")
df_s1 = pd.read_parquet(final_norm_paths['s1'])
s1_id_map = {eid: i for i, eid in enumerate(df_s1['entity_id'].values)}
s1_names = df_s1['name_core'].fillna('').values
s1_addrs = df_s1['addr_core'].fillna('').values
s1_houses = df_s1['house_number'].fillna('').values
s1_states = df_s1['state_code'].fillna('').values
del df_s1; gc.collect()

df_s2 = pd.read_parquet(final_norm_paths['s2'])
df_s3 = pd.read_parquet(final_norm_paths['s3'])
df_tgt = pd.concat([df_s2, df_s3], ignore_index=True)
del df_s2, df_s3; gc.collect()

tgt_id_map = {eid: i for i, eid in enumerate(df_tgt['entity_id'].values)}
tgt_names = df_tgt['name_core'].fillna('').values
tgt_addrs = df_tgt['addr_core'].fillna('').values
tgt_houses = df_tgt['house_number'].fillna('').values
tgt_states = df_tgt['state_code'].fillna('').values
tgt_is_s2 = df_tgt['entity_id'].str.startswith('S2').values.astype(np.int8)
del df_tgt; gc.collect()

print("  Record arrays loaded. Memory flat!")

# 3. Stream candidates and score with LightGBM in 2M chunks
scored_s1 = []
scored_cand = []
scored_probs = []

pf = pq.ParquetFile(str(cand_parquet_path))
chunk_size = 2_000_000
total_cand_rows = pf.metadata.num_rows
print(f"  Scoring {total_cand_rows:,} candidate pairs in chunks of {chunk_size:,}...")

t0_score = time.time()
for batch_idx, batch in enumerate(pf.iter_batches(batch_size=chunk_size)):
    df_chunk = batch.to_pandas()
    n_c = len(df_chunk)
    
    c_s1_ids = df_chunk['s1_id'].values
    c_cand_ids = df_chunk['candidate_id'].values
    c_sim_name = df_chunk['sim_name'].values.astype(np.float32)
    c_sim_addr = df_chunk['sim_addr'].values.astype(np.float32)
    c_r_name = df_chunk['route_name'].values.astype(np.int8)
    c_r_addr = df_chunk['route_addr'].values.astype(np.int8)
    del df_chunk; gc.collect()
    
    # Map to integer indices
    s1_idx = np.array([s1_id_map.get(sid, -1) for sid in c_s1_ids], dtype=np.int32)
    tgt_idx = np.array([tgt_id_map.get(cid, -1) for cid in c_cand_ids], dtype=np.int32)
    valid_mask = (s1_idx >= 0) & (tgt_idx >= 0)
    
    # Filter valid
    s1_idx = s1_idx[valid_mask]
    tgt_idx = tgt_idx[valid_mask]
    c_s1_ids = c_s1_ids[valid_mask]
    c_cand_ids = c_cand_ids[valid_mask]
    c_sim_name = c_sim_name[valid_mask]
    c_sim_addr = c_sim_addr[valid_mask]
    c_r_name = c_r_name[valid_mask]
    c_r_addr = c_r_addr[valid_mask]
    n_valid = len(s1_idx)
    
    # Feature 1-5: Vectors
    r_both = (c_r_name & c_r_addr).astype(np.int8)
    sim_prod = (c_sim_name * c_sim_addr).astype(np.float32)
    sim_max = np.maximum(c_sim_name, c_sim_addr).astype(np.float32)
    
    # Feature 6-13: String similarities (RapidFuzz C++)
    f_jw = np.zeros(n_valid, dtype=np.float32)
    f_name_jacc = np.zeros(n_valid, dtype=np.float32)
    f_name_exact = np.zeros(n_valid, dtype=np.int8)
    f_len_diff = np.zeros(n_valid, dtype=np.int16)
    f_first_tok = np.zeros(n_valid, dtype=np.int8)
    f_house = np.zeros(n_valid, dtype=np.int8)
    f_state = np.zeros(n_valid, dtype=np.int8)
    f_addr_jacc = np.zeros(n_valid, dtype=np.float32)
    
    for i in range(n_valid):
        s_i = s1_idx[i]
        t_i = tgt_idx[i]
        
        sn = s1_names[s_i]
        tn = tgt_names[t_i]
        if sn and tn:
            f_jw[i] = JaroWinkler.similarity(sn, tn)
            f_name_exact[i] = 1 if sn == tn else 0
            f_len_diff[i] = abs(len(sn) - len(tn))
            
            s_tok = sn.split()
            t_tok = tn.split()
            if s_tok and t_tok:
                f_first_tok[i] = 1 if s_tok[0] == t_tok[0] else 0
                s_set = set(s_tok)
                t_set = set(t_tok)
                union_len = len(s_set | t_set)
                f_name_jacc[i] = len(s_set & t_set) / union_len if union_len > 0 else 0.0
                
        # House & State
        sh = s1_houses[s_i]
        th = tgt_houses[t_i]
        if sh and th:
            f_house[i] = 1 if sh == th else -1
            
        ss = s1_states[s_i]
        ts = tgt_states[t_i]
        if ss and ts:
            f_state[i] = 1 if ss == ts else -1
            
        # Address Jaccard
        sa = s1_addrs[s_i]
        ta = tgt_addrs[t_i]
        if sa and ta:
            sa_set = set(sa.split())
            ta_set = set(ta.split())
            u_len = len(sa_set | ta_set)
            f_addr_jacc[i] = len(sa_set & ta_set) / u_len if u_len > 0 else 0.0

    # Build 19-column feature matrix
    X_chunk = np.column_stack([
        c_sim_name, c_sim_addr, c_r_name, c_r_addr, r_both,
        f_jw, f_name_jacc, f_name_exact, f_len_diff, f_first_tok,
        f_house, f_state, f_addr_jacc, tgt_is_s2[tgt_idx],
        sim_prod, sim_max, np.zeros(n_valid, dtype=np.float32), # gap_to_top placeholder
        np.zeros(n_valid, dtype=np.int16),                      # cand_rank placeholder
        np.full(n_valid, 20, dtype=np.int16)                   # n_cands placeholder
    ]).astype(np.float32)
    
    # Model ensemble prediction
    preds = np.zeros(n_valid, dtype=np.float32)
    for m in models:
        preds += m.predict(X_chunk)
    preds /= len(models)
    
    # Prune low scores immediately to keep memory flat (< 0.30 can never pass t_first=0.60, t_rest=0.575)
    keep_mask = (preds >= 0.30)
    scored_s1.extend(c_s1_ids[keep_mask])
    scored_cand.extend(c_cand_ids[keep_mask])
    scored_probs.extend(preds[keep_mask])
    
    del X_chunk, preds, keep_mask; gc.collect()
    print(f"    Batch {batch_idx+1}: processed {n_valid:,} pairs | Kept plausible: {len(scored_s1):,}")

print(f"Scoring complete in {time.time()-t0_score:.1f}s. Total plausible candidates: {len(scored_s1):,}")
del s1_id_map, tgt_id_map, s1_names, s1_addrs, s1_houses, s1_states; gc.collect()
del tgt_names, tgt_addrs, tgt_houses, tgt_states, tgt_is_s2; gc.collect()


In [ ]:
# ============================================================
# STAGE 5: SINGLE-PASS TWO-THRESHOLD DECISION RULE
# ============================================================

# Winning thresholds locked from NB 05 (ml5nbv1)
T_FIRST = 0.600
T_REST  = 0.575

print("\n" + "=" * 60)
print(f" APPLYING WINNING DECISION RULE: t_first={T_FIRST}, t_rest={T_REST}")
print("=" * 60)

# Group candidate scores by S1 entity
cands_by_s1 = {}
for sid, cid, prob in zip(scored_s1, scored_cand, scored_probs):
    sid = str(sid)
    if sid not in cands_by_s1:
        cands_by_s1[sid] = []
    cands_by_s1[sid].append((str(cid), float(prob)))

# Sort descending by score
for sid in cands_by_s1:
    cands_by_s1[sid].sort(key=lambda x: -x[1])

# Generate final predictions for all 1,732,544 test entities
final_test_matches = {}
all_test_s1_ids = s1_test_ids

n_singletons = 0
n_matched = 0
total_match_ids = 0

for sid in all_test_s1_ids:
    cands = cands_by_s1.get(sid, [])
    if not cands or cands[0][1] < T_FIRST:
        # Best candidate does not pass t_first -> singleton (empty string)
        final_test_matches[sid] = []
        n_singletons += 1
    else:
        # Accept top candidate
        accepted = [cands[0][0]]
        for cid, score in cands[1:]:
            if score >= T_REST:
                accepted.append(cid)
        final_test_matches[sid] = accepted
        n_matched += 1
        total_match_ids += len(accepted)

print(f"\nDecision Summary:")
print(f"  Total test entities:  {len(all_test_s1_ids):,}")
print(f"  Singletons (empty):   {n_singletons:,} ({n_singletons/len(all_test_s1_ids)*100:.1f}%)")
print(f"  Matched entities:     {n_matched:,} ({n_matched/len(all_test_s1_ids)*100:.1f}%)")
print(f"  Total match IDs:      {total_match_ids:,}")
print(f"  Avg matches / entity: {total_match_ids / max(n_matched, 1):.2f}")


In [ ]:
# ============================================================
# EXPORT FINAL SUBMISSION FILES
# ============================================================

matching_path = OUTPUT_DIR / 'matching_results.tsv'

print(f"Writing {matching_path}...")
with open(matching_path, 'w', encoding='utf-8') as f:
    f.write('source1_entity_id\tmatched_entity_ids\n')
    for sid in all_test_s1_ids:
        matches = final_test_matches.get(sid, [])
        match_str = ','.join(matches)
        f.write(f'{sid}\t{match_str}\n')

print(f"  Saved: {matching_path} ({matching_path.stat().st_size / 1e6:.1f} MB)")
print(f"  Saved: {cand_tsv_path} ({cand_tsv_path.stat().st_size / 1e6:.1f} MB)")

# Verify exact line counts
with open(matching_path, encoding='utf-8') as f:
    matching_rows = sum(1 for _ in f) - 1
with open(cand_tsv_path, encoding='utf-8') as f:
    cand_rows = sum(1 for _ in f) - 1

print(f"\nVerification of line counts:")
print(f"  matching_results.tsv: {matching_rows:,} (Expected: {len(all_test_s1_ids):,})")
print(f"  candidate_pairs.tsv:  {cand_rows:,} (Expected: {len(all_test_s1_ids):,})")

assert matching_rows == len(all_test_s1_ids), f"Mismatch in matching_results! Expected {len(all_test_s1_ids)}, got {matching_rows}"
assert cand_rows == len(all_test_s1_ids), f"Mismatch in candidate_pairs! Expected {len(all_test_s1_ids)}, got {cand_rows}"
print("✅ Row count verification PASSED.")


In [ ]:
# ============================================================
# OFFICIAL VALIDATION HARNESS EXECUTION
# ============================================================

print("\n" + "=" * 60)
print(" RUNNING SUBMISSION VALIDATION HARNESS")
print("=" * 60)

test_dir = TEST_S1_TSV.parent
print(f"  Testing against test dir: {test_dir}")

if VALIDATOR_SCRIPT and VALIDATOR_SCRIPT.exists():
    cmd = [
        sys.executable, str(VALIDATOR_SCRIPT),
        '--matching', str(matching_path),
        '--candidate', str(cand_tsv_path),
        '--test-dir', str(test_dir),
    ]
    print(f"  Running: {' '.join(cmd)}")
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print("STDERR:", res.stderr)
    if res.returncode == 0:
        print("\n🏆🏆 SUBMISSION VERIFIED: 100% VALID & READY FOR PORTAL 🏆🏆")
    else:
        print(f"\n⚠️ Validator exited with code {res.returncode}. Review output above.")
else:
    print("Running in-line validation checks...")
    # 1. Check header
    with open(matching_path, encoding='utf-8') as f:
        header = f.readline().strip().split('\t')
        assert header == ['source1_entity_id', 'matched_entity_ids'], f"Invalid header: {header}"
    # 2. Check no self-match
    with open(matching_path, encoding='utf-8') as f:
        next(f)
        for line in f:
            parts = line.strip().split('\t')
            sid = parts[0]
            if len(parts) > 1 and parts[1]:
                m_list = parts[1].split(',')
                for m in m_list:
                    assert not m.startswith('S1-'), f"Self-match error: {sid} matched {m}"
    print("✅ In-line validation checks PASSED: All formatting rules satisfied!")
